# ECoG Spike Detection

This notebook detects stimulation-evoked neural spikes from ECoG recordings using band-pass filtering, robust thresholding, raster plots, and average firing-rate curves. The course dataset is not bundled with the repository; place `ECoGData_AvailabletoStudents.mat` in the project root before running.

In [1]:
import numpy as np

# For loadmat, which imports data from Matlab files and stores them in memory as
#  scipy tensors.
import scipy.io as sio

# For graphing
import matplotlib.pyplot as plt
from pathlib import Path

# For filtering
from scipy import signal
from scipy.signal import butter, filtfilt, find_peaks


Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\Metal\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


In [2]:
# Place the MATLAB data file in this project root before running.
data_path = "ECoGData_AvailabletoStudents.mat"
data = sio.loadmat(data_path)

# Create references to the MATLAB arrays.
# This first one appears as data.Signal in the Matlab structure. e.g. data_ecog[0] is
#  data.Signal{1, 1}
data_ecog = data["data"]["Signal"][0][0][0]
stim_times = data["data"]["StimTimes"][0][0][0]
Fs = data["data"]["SamplingFreq"][0][0][0][0]

print("Signal: " + str(data_ecog.shape))  # Three stim conditions
print("Stim Times: " + str(stim_times.shape))  # in seconds!
print("SFreq: " + str(Fs))  # 24414 Hz


# For bandpass filtering consider these functions, but check your work!
#  https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.butter.html 
#  https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.filtfilt.html

# Convert MATLAB arrays to NumPy arrays
signals = [np.asarray(data_ecog[i]).ravel() for i in range(3)]
stims   = [np.asarray(stim_times[i]).ravel() for i in range(3)]


Signal: (3,)
Stim Times: (3,)
SFreq: 24414.0625


In [3]:
# Define time windows and spike parameters
# The window extends 0.5 s before and after the 1.0 s stimulus.
# Spikes will be binned in 20 ms intervals to compute firing rate.
pre, post, duration = 0.5, 0.5, 1.0
t0, t1 = -pre, duration + post
bin_size = 0.02
refractory = 1

# Create bin edges and centers for histogram
bin_edges = np.arange(t0, t1 + bin_size, bin_size)
centers = bin_edges[:-1] + bin_size/2

# Convert refractory time to no. of samples
dist = max(1, int(np.round(Fs * refractory / 1000.0)))


In [4]:
# Bandpass filter from 500–5000 Hz
# First-order filter applied with filtfilt()
cutoffs = [500 / (Fs / 2), 5000 / (Fs / 2)]
b, a = butter(1, cutoffs, btype='band')
bandpass = lambda x: filtfilt(b, a, x)


In [5]:
# Spike detection across all three stimulation patterns
# For each pattern:
# Filter the signal.
# Compute robust noise σ = median(|filtered|)/0.6745.
# Set threshold Thr = 3σ.
# Detect spikes in |segment| using find_peaks().
# Define the indices for current trial
# Extract signal segment and pad if near array edges
# Detect peaks using abs value and the threshold
# Convert sample indices to time
# Compute firing rate for histogram
rasters, fr_curves = [], []

for idx in range(3):
    filtered = bandpass(signals[idx])
    sigma = np.median(np.abs(filtered)) / 0.6745
    Thr   = 3.0 * sigma
    stim = stims[idx][:50]

    trial_spikes, trial_rates = [], []

    for t_start in stim:
        start_i = int((t_start + t0) * Fs)
        end_i   = int((t_start + t1) * Fs)

        pad_left  = max(0, -start_i)
        pad_right = max(0, end_i - len(filtered))
        s_idx     = max(0, start_i)
        e_idx     = min(end_i, len(filtered))

        segment = filtered[s_idx:e_idx]
        if pad_left or pad_right:
            segment = np.pad(segment, (pad_left, pad_right), mode='constant')

        seg_abs = np.abs(segment)
        peaks, _ = find_peaks(seg_abs, height=Thr, distance=dist)

        spikes = peaks / Fs + t0
        trial_spikes.append(spikes)

        counts, _ = np.histogram(spikes, bins=bin_edges)
        trial_rates.append(counts / bin_size)

    rasters.append(trial_spikes)
    fr_curves.append(np.vstack(trial_rates))


In [6]:
# Raster plots
# Displays spike timing across 50 trials per stimulation pattern.
# The shaded region represents the stimulation window [0, 1.0 s].
titles = ["Raster Plot for Signal 1", "Raster Plot for Signal 2", "Raster Plot for Signal 3"]
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

for i in range(3):
    ax = axes[i] 
    ax.axvspan(0, duration, alpha=0.2)

    # plot spike times for each trial
    for tr, spikes in enumerate(rasters[i]):
        ax.vlines(spikes, tr + 0.6, tr + 1.4, color="C0", lw=0.6)

    ax.set_ylim(0.5, len(rasters[i]) + 0.5)
    ax.set_ylabel("Trial")
    ax.set_title(titles[i])
    ax.grid(axis="x", linestyle=":", alpha=0.5)

axes[-1].set_xlim(t0, t1)
axes[-1].set_xlabel("Time relative")
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/raster_plot.png", dpi=300)
print("Saved: figures/raster_plot.png")


Saved: figures/raster_plot.png


In [7]:
# Average firing rate plots
# Each subplot shows the mean firing rate across 50 trials.
# The shaded gray area marks the stimulation period.
fig2, axes2 = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

for i in range(3):
    ax = axes2[i]
    mean_rate = fr_curves[i].mean(axis=0)
    ax.plot(centers, mean_rate, label=titles[i])

    ax.axvspan(0, duration, color="gray", alpha=0.2)
    ax.set_xlim(t0, t1)
    ax.set_ylabel("Firing rate")
    ax.grid(linestyle=":", alpha=0.5)
    ax.legend()
    ax.set_title(titles[i])

axes2[-1].set_xlabel("Time relative")
Path("figures").mkdir(exist_ok=True)
fig2.savefig("figures/average_firing_rate.png", dpi=300)
print("Saved: figures/average_firing_rate.png")


Saved: figures/average_firing_rate.png
